# Exploration: Overview for SCIP

<br>  

Language-agnostic overview of code indexed via SCIP (Semantic Code Intelligence Protocol).
Covers total graph size, SCIP-specific node counts, and internal type distribution by project and language.

### References
- [jqassistant](https://jqassistant.org)
- [Neo4j Python Driver](https://neo4j.com/docs/api/python-driver/current)
- [SCIP Index Format](https://sourcegraph.com/blog/announcing-scip)

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plot
from neo4j import GraphDatabase
from typing import cast, LiteralString

In [ ]:
# Please set the environment variable "NEO4J_INITIAL_PASSWORD" in your shell
# before starting jupyter notebook to provide the password for the user "neo4j".
# It is not recommended to hardcode the password into jupyter notebook for security reasons.

neo4j_password = os.environ.get("NEO4J_INITIAL_PASSWORD")
if not neo4j_password:
    raise ValueError("NEO4J_INITIAL_PASSWORD environment variable must be set")

driver = GraphDatabase.driver(uri="bolt://localhost:7687", auth=("neo4j", neo4j_password))
driver.verify_connectivity()

In [ ]:
def get_cypher_query_from_file(filename: str) -> str:
    with open(filename, encoding="utf-8") as file:
        return " ".join(file.readlines())


def query_cypher_to_data_frame(filename: str) -> pd.DataFrame:
    records, _, keys = driver.execute_query(cast(LiteralString, get_cypher_query_from_file(filename)))
    return pd.DataFrame([r.values() for r in records], columns=keys)


main_color_map: str = "nipy_spectral"

In [ ]:
%%html
<style>
/* CSS style for smaller dataframe tables. */
.dataframe th {
    font-size: 8px;
}
.dataframe td {
    font-size: 8px;
}
</style>

## Overview

### Table 1 - Size

Total node and relationship counts plus SCIP-specific counts: artifacts, modules, internal types, external types, and total reference count across all DEPENDS_ON edges from internal types.

In [ ]:
overview_size = query_cypher_to_data_frame("../queries/overview/Overview_size_for_Scip.cypher")
overview_size

## Types per Project

### Table 2a - Types per project (raw, top 30)

Shows the largest (by type count) project-language combinations.
The `projectName` value is the SCIP module identifier (for example, `mvn/org.example/core`).
The full data can be found in the CSV report `Number_of_types_per_project_for_Scip`.

In [ ]:
types_per_project = query_cypher_to_data_frame("../queries/overview/Number_of_types_per_project_for_Scip.cypher")
types_per_project.head(30)

### Table 2b - Types per project grouped by language

Each row is a project, each column is a language, each value is the number of internal types.
The source data for this aggregated table can be found in the CSV report `Number_of_types_per_project_for_Scip`.

In [ ]:
types_per_project_grouped = types_per_project.pivot(
    index="projectName", columns="language", values="numberOfTypes"
)

types_per_project_grouped.fillna(0, inplace=True)

types_per_project_grouped["Total"] = types_per_project_grouped.sum(axis=1)
types_per_project_grouped.sort_values(by="Total", ascending=False, inplace=True)
types_per_project_grouped.drop("Total", axis=1, inplace=True)

column_totals = types_per_project_grouped.sum()
types_per_project_grouped = types_per_project_grouped[
    column_totals.sort_values(ascending=False).index
]

types_per_project_grouped = types_per_project_grouped.astype(int)

types_per_project_grouped.head(30)

### Table 2b Chart 1 - Top 30 projects by type count (stacked by language)

In [ ]:
if types_per_project_grouped.empty:
    print("No SCIP type data found — skipping chart.")
else:
    figure, axis = plot.subplots(figsize=(9, 6))
    types_per_project_grouped.head(30).plot(
        ax=axis, kind="bar", stacked=True, cmap=main_color_map
    )
    axis.set_title("Top 30 types per project")
    axis.set_xlabel("Project")
    axis.set_ylabel("Types")
    plot.show()